In [14]:
import csv
from collections import Counter

def load_cdrs(path, length=8):
    """
    Загружает CDR1/2 AA из файла формата:
    cdr1_heavy, cdr1_aa_heavy, cdr2_heavy, cdr2_aa_heavy, v_call_heavy
    Фильтруем только те, что длиной length.
    """
    cdr1_list = []
    cdr2_list = []

    with open(path, newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            cdr1 = row["cdr1_aa_heavy"].strip()
            cdr2 = row["cdr2_aa_heavy"].strip()

            if len(cdr1) != length or len(cdr2) != length:
                continue

            cdr1_list.append(cdr1)
            cdr2_list.append(cdr2)

    return cdr1_list, cdr2_list


def compute_position_freqs(sequences, length=8):
    """
    sequences: список AA-последовательностей одинаковой длины.
    Возвращает список длины L:
      pos_freqs[i] = {AA: frequency}
    """
    pos_counts = [Counter() for _ in range(length)]

    for seq in sequences:
        for i, aa in enumerate(seq):
            pos_counts[i][aa] += 1

    pos_freqs = []
    for i in range(length):
        total = sum(pos_counts[i].values())
        freqs = {aa: count / total for aa, count in pos_counts[i].items()}
        pos_freqs.append(freqs)

    return pos_freqs


cdr1_list, cdr2_list = load_cdrs("ighv3-21_cdr_both_length8.csv", length=8)

cdr1_pos_freqs = compute_position_freqs(cdr1_list, length=8)
cdr2_pos_freqs = compute_position_freqs(cdr2_list, length=8)

print("Пример частот на позиции 1 для CDR1:", cdr1_pos_freqs[0])
print("Пример частот на позиции 1 для CDR2:", cdr2_pos_freqs[0])

Пример частот на позиции 1 для CDR1: {'G': 0.9731721295816163, 'D': 0.002510848503043039, 'A': 0.004530444038099397, 'E': 0.011244234600584045, 'H': 8.187549466444693e-05, 'N': 0.0003002101471029721, 'Q': 0.0008733386097541006, 'V': 0.0002456264839933408, 'K': 0.0012008405884118883, 'R': 0.00382085641767419, 'T': 0.0013372997461859664, 'S': 0.0006277121257607598, 'L': 2.7291831554815644e-05, 'I': 2.7291831554815644e-05}
Пример частот на позиции 1 для CDR2: {'I': 0.9642204088316367, 'V': 0.013209246472530772, 'L': 0.006822957888703911, 'Y': 0.00010916732621926257, 'A': 0.0001910428208837095, 'T': 0.0031931442919134303, 'M': 0.00826942496110914, 'S': 0.001310007914631151, 'N': 0.00013645915777407821, 'F': 0.0023470975137141453, 'K': 2.7291831554815644e-05, 'D': 5.458366310963129e-05, 'H': 2.7291831554815644e-05, 'C': 5.458366310963129e-05, 'G': 2.7291831554815644e-05}


In [4]:
import csv
from collections import Counter, defaultdict

In [15]:
from itertools import product


GENETIC_CODE = {
    # Phenylalanine
    "TTT": "F", "TTC": "F",
    # Leucine
    "TTA": "L", "TTG": "L",
    "CTT": "L", "CTC": "L", "CTA": "L", "CTG": "L",
    # Isoleucine
    "ATT": "I", "ATC": "I", "ATA": "I",
    # Methionine (start)
    "ATG": "M",
    # Valine
    "GTT": "V", "GTC": "V", "GTA": "V", "GTG": "V",
    # Serine
    "TCT": "S", "TCC": "S", "TCA": "S", "TCG": "S",
    # Proline
    "CCT": "P", "CCC": "P", "CCA": "P", "CCG": "P",
    # Threonine
    "ACT": "T", "ACC": "T", "ACA": "T", "ACG": "T",
    # Alanine
    "GCT": "A", "GCC": "A", "GCA": "A", "GCG": "A",
    # Tyrosine
    "TAT": "Y", "TAC": "Y",
    # Histidine
    "CAT": "H", "CAC": "H",
    # Glutamine
    "CAA": "Q", "CAG": "Q",
    # Asparagine
    "AAT": "N", "AAC": "N",
    # Lysine
    "AAA": "K", "AAG": "K",
    # Aspartic Acid
    "GAT": "D", "GAC": "D",
    # Glutamic Acid
    "GAA": "E", "GAG": "E",
    # Cysteine
    "TGT": "C", "TGC": "C",
    # Tryptophan
    "TGG": "W",
    # Arginine
    "CGT": "R", "CGC": "R", "CGA": "R", "CGG": "R",
    "AGA": "R", "AGG": "R",
    # Serine (AGY)
    "AGT": "S", "AGC": "S",
    # Glycine
    "GGT": "G", "GGC": "G", "GGA": "G", "GGG": "G",

    # Stop codons
    "TAA": "*", "TAG": "*", "TGA": "*",
}

IUPAC_NT = {
    "A": {"A"},
    "C": {"C"},
    "G": {"G"},
    "T": {"T"},
    "R": {"A", "G"},
    "Y": {"C", "T"},
    "S": {"G", "C"},
    "W": {"A", "T"},
    "K": {"G", "T"},
    "M": {"A", "C"},
    "B": {"C", "G", "T"},
    "D": {"A", "G", "T"},
    "H": {"A", "C", "T"},
    "V": {"A", "C", "G"},
    "N": {"A", "C", "G", "T"},
}

In [16]:
def expand_degenerate_codon(deg_codon):
    """
    deg_codon: строка из 3 IUPAC-символов, например 'NNK'.
    Возвращает список конкретных триплетов, например ['AAG', 'GAG', ...].
    """
    assert len(deg_codon) == 3
    pools = []
    for ch in deg_codon:
        if ch not in IUPAC_NT:
            raise ValueError(f"Unknown IUPAC symbol: {ch}")
        pools.append(sorted(IUPAC_NT[ch]))
    concrete_codons = ["".join(c) for c in product(*pools)]
    return concrete_codons

def aa_distribution_for_degenerate(deg_codon, ignore_stop=True):
    """
    Возвращает (aa_freq_dict, has_stop).
    aa_freq_dict: {AA: freq}, freq суммируются в 1.0 (если не считать стопы).
    """
    codons = expand_degenerate_codon(deg_codon)
    aa_counts = Counter()
    stop_count = 0
    for codon in codons:
        aa = GENETIC_CODE.get(codon)
        if aa is None:
            continue
        if aa == "*":
            stop_count += 1
        else:
            aa_counts[aa] += 1

    total_non_stop = sum(aa_counts.values())
    total_all = total_non_stop + stop_count

    if total_all == 0:
        return {}, stop_count > 0

    if ignore_stop:
        if total_non_stop == 0:
            return {}, True
        freqs = {aa: c / total_non_stop for aa, c in aa_counts.items()}
    else:
        freqs = {aa: c / total_all for aa, c in aa_counts.items()}
        if stop_count > 0:
            freqs["*"] = stop_count / total_all

    return freqs, (stop_count > 0)

def generate_all_degenerate_codons():
    symbols = list(IUPAC_NT.keys())
    candidates = []
    for a, b, c in product(symbols, repeat=3):
        deg = a + b + c
        aa_freqs, has_stop = aa_distribution_for_degenerate(deg)
        if has_stop:
            continue
        if not aa_freqs:
            continue
        candidates.append((deg, aa_freqs))
    return candidates

DEGENERATE_CODON_CANDIDATES = generate_all_degenerate_codons()
print(f"Total candidates (no stops): {len(DEGENERATE_CODON_CANDIDATES)}")

Total candidates (no stops): 2351


In [17]:
def score_degenerate_codon(target_freqs, cand_freqs, lambda_extra=1.0):
    """
    Чем МЕНЬШЕ score, тем лучше.
    - первая часть: L2-норма разницы распределений по пересекающимся AA
    - вторая часть: штраф за 'лишние' аминокислоты
    """
    all_aas = set(target_freqs.keys()) | set(cand_freqs.keys())
    l2 = 0.0
    extra_penalty = 0.0

    for aa in all_aas:
        t = target_freqs.get(aa, 0.0)
        c = cand_freqs.get(aa, 0.0)
        l2 += (t - c) ** 2
        if t == 0.0 and c > 0.0:
            extra_penalty += c

    return l2 + lambda_extra * extra_penalty

In [18]:
def choose_best_degenerate_codon_for_position(target_freqs,
                                              candidates=DEGENERATE_CODON_CANDIDATES,
                                              lambda_extra=1.0):
    """
    target_freqs: словарь {AA: freq} для одной позиции.
    Возвращает (best_deg_codon, best_score, best_dist).
    best_dist: распределение AA для этого deg-кодона.
    """
    best_deg = None
    best_score = float("inf")
    best_dist = None

    for deg, aa_dist in candidates:
        score = score_degenerate_codon(target_freqs, aa_dist, lambda_extra=lambda_extra)
        if score < best_score:
            best_score = score
            best_deg = deg
            best_dist = aa_dist

    return best_deg, best_score, best_dist

In [19]:
def design_degenerate_codons_for_cdr(pos_freqs, lambda_extra=1.0):
    """
    pos_freqs: список длины L, pos_freqs[i] = {AA: freq} на позиции i.
    Возвращает список длины L с deg-кодонами.
    """
    L = len(pos_freqs)
    deg_codons = []
    details = []

    for i in range(L):
        target = pos_freqs[i]
        best_deg, best_score, best_dist = choose_best_degenerate_codon_for_position(
            target, lambda_extra=lambda_extra
        )
        deg_codons.append(best_deg)
        details.append({
            "position": i + 1,
            "target_freqs": target,
            "chosen_deg": best_deg,
            "deg_aa_dist": best_dist,
            "score": best_score,
        })
    return deg_codons, details

cdr1_deg_codons, cdr1_details = design_degenerate_codons_for_cdr(cdr1_pos_freqs)
cdr2_deg_codons, cdr2_details = design_degenerate_codons_for_cdr(cdr2_pos_freqs)

print("CDR1 degenerate codons (8 positions):", cdr1_deg_codons)
print("CDR2 degenerate codons (8 positions):", cdr2_deg_codons)

CDR1 degenerate codons (8 positions): ['GGA', 'TTC', 'ACA', 'TTC', 'AGC', 'AGB', 'TAC', 'AGC']
CDR2 degenerate codons (8 positions): ['ATA', 'AGC', 'AGC', 'AGC', 'AGC', 'WSC', 'TAC', 'ATA']


In [25]:
import numpy as np
from scipy.spatial.distance import cdist
from sklearn.cluster import KMeans
import csv
from collections import Counter, defaultdict

def find_consensus_sequences(cdr_sequences, num_consensus=3):
    """
    Находит консенсусные последовательности, близкие к большинству в выборке
    """
    aa_to_idx = {aa: i for i, aa in enumerate('ACDEFGHIKLMNPQRSTVWY')}

    encoded_seqs = []
    for seq in cdr_sequences:
        encoded = []
        for aa in seq:
            vec = [0] * len(aa_to_idx)
            if aa in aa_to_idx:
                vec[aa_to_idx[aa]] = 1
            encoded.extend(vec)
        encoded_seqs.append(encoded)

    kmeans = KMeans(n_clusters=min(num_consensus, len(encoded_seqs)), random_state=42)
    labels = kmeans.fit_predict(encoded_seqs)

    consensus_seqs = []
    for center in kmeans.cluster_centers_:
        distances = []
        for i, encoded in enumerate(encoded_seqs):
            dist = np.linalg.norm(np.array(encoded) - center)
            distances.append((dist, i))

        distances.sort()
        closest_idx = distances[0][1]
        consensus_seqs.append(cdr_sequences[closest_idx])

    return consensus_seqs

def optimize_degenerate_pool(target_freqs_list, consensus_sequences, max_pool_size=5):
    """
    Оптимизирует пул вырожденных олигонуклеотидов для покрытия целевых частот
    """
    best_pools = []

    for pos_idx, target_freqs in enumerate(target_freqs_list):
        codon_scores = []

        for deg_codon, aa_dist in DEGENERATE_CODON_CANDIDATES:
            coverage_score = 0
            for aa, target_freq in target_freqs.items():
                actual_freq = aa_dist.get(aa, 0)
                coverage_score += min(actual_freq, target_freq)

            extra_penalty = sum(aa_dist.get(aa, 0) for aa in aa_dist
                              if aa not in target_freqs or target_freqs.get(aa, 0) == 0)

            total_score = coverage_score - 0.5 * extra_penalty
            codon_scores.append((deg_codon, aa_dist, total_score))

        codon_scores.sort(key=lambda x: x[2], reverse=True)

        selected_codons = []
        covered_aas = set()

        for deg_codon, aa_dist, score in codon_scores:
            if len(selected_codons) >= max_pool_size:
                break

            new_aas = set(aa for aa, freq in aa_dist.items()
                         if freq > 0.1 and aa not in covered_aas)

            if new_aas or not selected_codons:
                selected_codons.append((deg_codon, aa_dist))
                covered_aas.update(aa for aa, freq in aa_dist.items() if freq > 0.1)

        best_pools.append(selected_codons)

    return best_pools

def calculate_library_coverage(optimized_pools, original_sequences):
    """
    Оценивает покрытие оригинальных последовательностей оптимизированным пулом
    """
    coverage_count = 0
    total_sequences = len(original_sequences)

    for seq in original_sequences:
        covered = True
        for pos, aa in enumerate(seq):
            pos_covered = False
            for deg_codon, aa_dist in optimized_pools[pos]:
                if aa_dist.get(aa, 0) > 0:
                    pos_covered = True
                    break
            if not pos_covered:
                covered = False
                break

        if covered:
            coverage_count += 1

    return coverage_count / total_sequences

def expand_degenerate_sequence(deg_sequence):
    """
    Расширяет вырожденную нуклеотидную последовательность в все возможные конкретные последовательности
    """
    nucleotide_sets = []
    for nt in deg_sequence:
        nucleotide_sets.append(IUPAC_NT.get(nt, {nt}))

    concrete_sequences = []
    for combo in product(*nucleotide_sets):
        concrete_sequences.append(''.join(combo))

    return concrete_sequences

def design_optimal_degenerate_library(cdr1_list, cdr2_list, max_pool_size=3):
    """
    Основная функция для дизайна оптимальной вырожденной библиотеки
    """
    cdr1_consensus = find_consensus_sequences(cdr1_list, num_consensus=2)
    print(f"Консенсусные последовательности CDR1: {cdr1_consensus}")

    cdr1_pos_freqs = compute_position_freqs(cdr1_list, length=8)
    cdr1_optimized = optimize_degenerate_pool(cdr1_pos_freqs, cdr1_consensus, max_pool_size)

    cdr2_consensus = find_consensus_sequences(cdr2_list, num_consensus=2)
    print(f"Консенсусные последовательности CDR2: {cdr2_consensus}")

    cdr2_pos_freqs = compute_position_freqs(cdr2_list, length=8)
    cdr2_optimized = optimize_degenerate_pool(cdr2_pos_freqs, cdr2_consensus, max_pool_size)

    cdr1_coverage = calculate_library_coverage(cdr1_optimized, cdr1_list)
    cdr2_coverage = calculate_library_coverage(cdr2_optimized, cdr2_list)

    print(f"Покрытие CDR1: {cdr1_coverage:.2%}")
    print(f"Покрытие CDR2: {cdr2_coverage:.2%}")

    return {
        'cdr1_pools': cdr1_optimized,
        'cdr2_pools': cdr2_optimized,
        'cdr1_consensus': cdr1_consensus,
        'cdr2_consensus': cdr2_consensus,
        'coverage': (cdr1_coverage, cdr2_coverage)
    }

cdr1_list, cdr2_list = load_cdrs("ighv3-21_cdr_both_length8.csv", length=8)

print(f"Загружено CDR1: {len(cdr1_list)} последовательностей")
print(f"Загружено CDR2: {len(cdr2_list)} последовательностей")

results = design_optimal_degenerate_library(cdr1_list, cdr2_list, max_pool_size=3)

print("\nCDR1 (позиции 1-8):")
for pos, pool in enumerate(results['cdr1_pools']):
    print(f"Позиция {pos+1}: {[codon for codon, _ in pool]}")

print("\nCDR2 (позиции 1-8):")
for pos, pool in enumerate(results['cdr2_pools']):
    print(f"Позиция {pos+1}: {[codon for codon, _ in pool]}")

def analyze_diversity_coverage(original_freqs, designed_pools):
    """
    Анализирует насколько хорошо дизайн покрывает оригинальное разнообразие
    """
    for region in ['CDR1', 'CDR2']:
        if region == 'CDR1':
            orig_freqs = original_freqs[0]
            pools = results['cdr1_pools']
        else:
            orig_freqs = original_freqs[1]
            pools = results['cdr2_pools']

        print(f"\n{region}:")
        for pos in range(8):
            target_aas = {aa for aa, freq in orig_freqs[pos].items() if freq > 0.01}
            designed_aas = set()
            for codon, aa_dist in pools[pos]:
                designed_aas.update(aa for aa, freq in aa_dist.items() if freq > 0.01)

            coverage = len(target_aas & designed_aas) / len(target_aas) if target_aas else 1.0
            extra = len(designed_aas - target_aas)

            print(f"  Позиция {pos+1}: покрытие {coverage:.1%}, лишних AA: {extra}")

analyze_diversity_coverage([cdr1_pos_freqs, cdr2_pos_freqs], results)

Загружено CDR1: 36641 последовательностей
Загружено CDR2: 36641 последовательностей
Консенсусные последовательности CDR1: ['GFTFSTYT', 'GFTFSSYS']
Консенсусные последовательности CDR2: ['ISSSSSYI', 'ISSSSTYI']
Покрытие CDR1: 65.75%
Покрытие CDR2: 58.60%

CDR1 (позиции 1-8):
Позиция 1: ['GGA', 'GRS', 'GSA']
Позиция 2: ['TTC', 'TTB', 'WTC']
Позиция 3: ['ACA', 'ASS', 'AYR']
Позиция 4: ['TTC', 'TTB', 'WTC']
Позиция 5: ['AGC', 'AGB', 'ARC']
Позиция 6: ['AGB', 'WSC', 'ARC']
Позиция 7: ['TAC', 'YAC', 'TWC']
Позиция 8: ['AGC', 'AGB', 'WSC']

CDR2 (позиции 1-8):
Позиция 1: ['ATA', 'ATN', 'RTA']
Позиция 2: ['AGC', 'AGB', 'WSC']
Позиция 3: ['AGC', 'AGB', 'RGC']
Позиция 4: ['AGC', 'AGB', 'RGC']
Позиция 5: ['AGC', 'AGB', 'RGC']
Позиция 6: ['AGB', 'WSC', 'ARC']
Позиция 7: ['TAC', 'TWC', 'YAC']
Позиция 8: ['ATA', 'ATN', 'AYA']

CDR1:
  Позиция 1: покрытие 100.0%, лишних AA: 2
  Позиция 2: покрытие 100.0%, лишних AA: 1
  Позиция 3: покрытие 50.0%, лишних AA: 2
  Позиция 4: покрытие 100.0%, лишних AA: 